In [1]:
# Install TensorFlow if needed
!pip install tensorflow

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Dataset
text = """
artificial intelligence systems learn patterns from data
sequence models process information step by step
recurrent neural networks are useful for sequence prediction
lstm networks handle long term dependencies
deep learning models improve sequence learning
generative models create new samples from learned patterns
language models predict the next word in a sentence
sequence generation is used in chatbots and assistants
machine learning helps computers learn automatically
training data improves model accuracy
neural networks simulate human brain structures
optimization algorithms improve learning efficiency
technology is transforming modern education
online learning platforms use artificial intelligence
students benefit from intelligent tutoring systems
automation improves productivity and decision making
"""

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

# Create sequences
input_sequences = []

for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)

# Padding
max_seq_len = max([len(x) for x in input_sequences])

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')
)

# Split input/output
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encoding
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Model
model = Sequential([
    Embedding(total_words, 64, input_length=X.shape[1]),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X, y, epochs=200, verbose=1)

# Generate text
seed_text = "artificial intelligence"
next_words = 5

for _ in range(next_words):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

    predicted = np.argmax(model.predict(token_list), axis=-1)[0]

    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted:
            output_word = word
            break

    seed_text += " " + output_word

print("Generated Text:")
print(seed_text)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.0341 - loss: 4.3565
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0568 - loss: 4.3458    
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0568 - loss: 4.3360
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0568 - loss: 4.3241
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0682 - loss: 4.3054
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0682 - loss: 4.2768
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0682 - loss: 4.2340
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0568 - loss: 4.1559    
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0455 - loss: 4.1095
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0455 - loss: 4.0968
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0455 - loss: 4.0598
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0

In [3]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, Dense, LayerNormalization, MultiHeadAttention
from tensorflow.keras import Model

# Dataset
sentences = [
    "artificial intelligence systems learn patterns from data",
    "sequence models process information step by step",
    "recurrent neural networks are useful for sequence prediction",
    "lstm networks handle long term dependencies"
]

# Vectorization
vectorize = TextVectorization(output_sequence_length=10)
vectorize.adapt(sentences)

vocab_size = len(vectorize.get_vocabulary())

# Convert text
x_train = vectorize(sentences) # Renamed to x_train

# Simple transformer block
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.norm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.norm2(out1 + ffn_output)

# Model
inputs = tf.keras.Input(shape=(10,))
x_model_output_tensor = Embedding(vocab_size, 32)(inputs) # Renamed
x_model_output_tensor = TransformerBlock(32, 2, 32)(x_model_output_tensor)
x_model_output_tensor = Dense(vocab_size, activation="softmax")(x_model_output_tensor)

model = Model(inputs, x_model_output_tensor) # Use the correct tensor here

model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")

# Dummy training target
# For sparse_categorical_crossentropy with sequence-to-sequence, target should be integer labels
y_train = x_train # Use input sequences as dummy targets

# Train
model.fit(x_train, y_train, epochs=20)

print("Transformer model trained successfully")

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - loss: 3.7954
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 3.2957
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 2.8203
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 2.4332
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 2.1698
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 1.9798
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 1.8256
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 1.6909
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 1.5680
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 1.4539
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 1.3480
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 1.2496
Epoch 13/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 1.1572
Epoch 14/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 1.0692
Epoch 15/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.9865
Epoch 16/20
1/1 ━━━━━━━━━━━━━━━━━━━━